In [ ]:
!pip install -q \
  transformers \
  huggingface_hub \
  evaluate

!pip install -U bitsandbytes

In [ ]:
import re
import torch
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import hf_hub_download, login

Upload your PDF…


Saving qa_base.pdf to qa_base.pdf


In [ ]:
# -----------------------------
# 0) Helpers / configuration
# -----------------------------
LABELS = {"negative", "neutral", "positive"}
SPLIT = re.compile(r'(?<=[.!?])\s+')

def label_to_score(lbl: str) -> float:
    lbl = lbl.strip().lower()
    if lbl == "positive": return +1.0
    if lbl == "neutral":  return  0.0
    if lbl == "negative": return -1.0
    # fallback if model outputs unexpected token
    return 0.0

Segments detected: 68


In [ ]:
# -----------------------------
# 1) Single-turn classification
#    (discrete label; your TinyLLaMA generate)
# -----------------------------
def classify_label(model, tokenizer, text: str, device: str) -> str:
    instruction = (
        "### Instruction:\n"
        "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
        "### Bewertung:\n"
    )
    answer_prefix = "\n\n### Antwort:\n"
    prompt = instruction + text + answer_prefix

    # fast & safe defaults
    tokenizer.truncation_side = "left"
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=2046, padding=False).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2,          # we only expect one label token
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # robust extraction
    answer_part = decoded.split("### Antwort:")[-1] if "### Antwort:" in decoded else decoded
    label = (answer_part.strip().split() or [""])[0].lower()
    return label if label in LABELS else "neutral"

In [ ]:
# -----------------------------
# 2) Message-level sentiment
#    (multi-sentence → length-weighted average of label scores)
# -----------------------------
def sentiment_score(model, tokenizer, device, text: str, max_sents: int = 12) -> float:
    sents = [s.strip() for s in SPLIT.split(text) if s.strip()]
    if not sents:
        return 0.0
    # cap for speed
    sents = sents[:max_sents]

    scores, lengths = [], []
    for s in sents:
        lbl = classify_label(model, tokenizer, s, device)
        scores.append(label_to_score(lbl))
        lengths.append(len(s))

    w = sum(lengths) or 1
    return sum(sc * L for sc, L in zip(scores, lengths)) / w  # [-1..1]

Filtered domain_qa.jsonl → domain_qa_clean.jsonl (193 remaining lines)


In [ ]:
# -----------------------------
# 3) Smoothing + trend reversal
# -----------------------------
class EMA: # Exponential Moving Average
    def __init__(self, alpha=0.35):
        self.a, self.v = alpha, None
    def update(self, x: float) -> float:
        self.v = x if self.v is None else (self.a * x + (1 - self.a) * self.v) # The influence of old data points decays exponentially over time.
        return self.v

class Trend:  # flips only on sustained moves to avoid jitter
    def __init__(self, up_thr=+0.06, down_thr=-0.06, sustain=2):
        self.state, self.prev = "flat", None     # "up" | "down" | "flat"
        self.up_thr, self.down_thr, self.sustain = up_thr, down_thr, sustain
        self.count = 0
    def update(self, smoothed: float):
        event = None
        if self.prev is None:
            self.prev = smoothed
            return self.state, event
        delta = smoothed - self.prev
        self.prev = smoothed

        direction = "flat"
        if   delta >= self.up_thr:   direction = "up"
        elif delta <= self.down_thr: direction = "down"

        if direction == "flat" or direction == self.state:
            self.count = 0
            if direction == "flat": self.state = "flat"
            return self.state, event

        # different direction trend -> require sustained evidence
        self.count += 1
        if self.count >= self.sustain:
            self.state = direction
            self.count = 0
            event = f"reversal_to_{direction}"
        return self.state, event


In [ ]:
# -----------------------------
# 4) Orchestrator
# -----------------------------
@dataclass
class TurnInfo:
    who: str           # "human" | "bot"
    text: str
    s_raw: float       # per-turn score in [-1..1]
    s_ema: float       # smoothed
    trend: str         # "up"|"down"|"flat"
    event: str | None  # "reversal_to_up"/"reversal_to_down"/None

class SentimentTracker:
    def __init__(self, model, tokenizer, device,
                 alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2):
        self.model, self.tokenizer, self.device = model, tokenizer, device
        self.ema = EMA(alpha)
        self.trend = Trend(up_thr, down_thr, sustain)
        self.history: list[TurnInfo] = []

    def step(self, who: str, text: str) -> TurnInfo:
        raw = sentiment_score(self.model, self.tokenizer, self.device, text)  # handles single→multi sentence
        sm  = self.ema.update(raw)
        tr, ev = self.trend.update(sm)
        info = TurnInfo(who, text, raw, sm, tr, ev)
        self.history.append(info)
        return info

In [ ]:
device = "cuda" # Used in colab

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-sentiment-4bit-v7"
subfolder = "adapters/epoch_005"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

In [ ]:
# 4-bit quantization config
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# Load base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quant_config,
    device_map="auto",
)

# Attach LoRA adapters
lora_model = PeftModel.from_pretrained(base_model, lora_repo_id, subfolder=subfolder)
#lora_model = PeftModel.from_pretrained(base_model, lora_repo_id)

lora_model = lora_model.to(device)
lora_model.eval()

In [ ]:
# -----------------------------
# 5) Example
# -----------------------------

tracker = SentimentTracker(lora_model, tokenizer, device, alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2)

dialog = [
    # --- Negative / frustrated start ---
    ("human", "Ich bin wirklich enttäuscht. Seit Tagen habe ich keine Antwort bekommen."),
    ("bot",   "Es tut mir leid, dass Sie warten mussten. Ich sehe mir das sofort an."),
    ("human", "Das ist nicht das erste Mal, dass so etwas passiert. Ich bin langsam echt genervt."),
    ("bot",   "Ich verstehe Ihren Ärger gut. Wir versuchen, das künftig zu vermeiden."),
    ("human", "Na gut... wenigstens kümmern Sie sich diesmal."),

    # --- Neutral / clarifying phase ---
    ("bot",   "Können Sie mir bitte Ihre Kundennummer nennen, damit ich den Fall prüfen kann?"),
    ("human", "Ja, die Nummer ist 34821. Ich hoffe, das hilft."),
    ("bot",   "Vielen Dank. Einen Moment bitte, ich überprüfe die Daten."),

    # --- Positive resolution phase ---
    ("bot",   "Ich habe den Vorgang gefunden und kann ihn heute noch abschließen."),
    ("human", "Oh, das klingt gut. Danke für die schnelle Hilfe."),
    ("bot",   "Sehr gerne! Ich freue mich, wenn ich Ihnen weiterhelfen konnte."),
    ("human", "Alles klar, jetzt bin ich zufrieden. Einen schönen Tag noch!"),
]

for i, (who, text) in enumerate(dialog, 1):
    info = tracker.step(who, text)
    flag = f"  -> {info.event}" if info.event else ""
    print(f"{i:02d} [{who}] s_raw={info.s_raw:+.2f}  s_ema={info.s_ema:+.2f}  trend={info.trend}{flag}")

# Use tracker.history to plot the Sentiment-Verlauf and mark trend reversals.
